# UAP Video Frame Analysis

This notebook creates a reproducible workflow for processing UAP (*unidentified anomalous phenomena*) videos. The main function receives a video file, reads its FPS, exports every frame as an image, and compares the apparent target motion with the global scene/camera motion.

The final classification is heuristic: it points to the most likely cause of a target disappearing from the scene, using evidence such as residual object speed, residual acceleration, camera motion change, and target proximity to the frame edge.

For targeting-pod, drone, or fighter footage, the camera-motion estimator masks the central reticle/crosshair and the tracked target area. It also builds an automatic textured-background mask that rejects black overlays, saturated HUD marks, and flat regions. This prevents fixed overlays or the centered target from making the camera look artificially stationary.
For fast pans over textured terrain, the primary camera estimator is phase correlation on masked background texture. Block matching, dense optical flow, and ORB features are kept as fallbacks.

## Dependencies

If needed, install the libraries below in the notebook environment:

```bash
pip install opencv-python numpy pandas matplotlib
```

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

try:
    import cv2
except ImportError as exc:
    raise ImportError(
        "Install the dependencies with: pip install opencv-python numpy pandas matplotlib"
    ) from exc


def _safe_fps(cap: cv2.VideoCapture, fallback: float = 30.0) -> float:
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    if not np.isfinite(fps) or fps <= 0:
        return fallback
    return fps


def _validate_box(box: tuple[int, int, int, int] | None, width: int, height: int, name: str):
    if box is None:
        return None
    x, y, w, h = [int(v) for v in box]
    if w <= 0 or h <= 0:
        raise ValueError(f"{name} must have positive width and height: {box}")
    if x < 0 or y < 0 or x + w > width or y + h > height:
        raise ValueError(f"{name} is outside the {width}x{height} frame: {box}")
    return (x, y, w, h)


def _validate_boxes(
    boxes: list[tuple[int, int, int, int]] | None,
    width: int,
    height: int,
    name: str,
) -> list[tuple[int, int, int, int]]:
    if boxes is None:
        return []
    return [_validate_box(box, width, height, f"{name}[{idx}]") for idx, box in enumerate(boxes)]


def _crop(frame: np.ndarray, roi: tuple[int, int, int, int] | None):
    if roi is None:
        return frame, (0, 0)
    x, y, w, h = roi
    return frame[y : y + h, x : x + w], (x, y)


def _to_gray(frame: np.ndarray) -> np.ndarray:
    if frame.ndim == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    return gray


def _identity_affine() -> np.ndarray:
    return np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], dtype=np.float32)


def _transform_point(point: np.ndarray, affine: np.ndarray) -> np.ndarray:
    point = np.asarray(point, dtype=np.float32).reshape(1, 1, 2)
    return cv2.transform(point, affine).reshape(2).astype(float)


def _edge_distance(point: np.ndarray, width: int, height: int) -> float:
    x, y = float(point[0]), float(point[1])
    return float(min(x, y, width - 1 - x, height - 1 - y))


def _clip_box_to_shape(
    box: tuple[float, float, float, float],
    width: int,
    height: int,
) -> tuple[int, int, int, int] | None:
    x, y, w, h = box
    x1 = max(0, int(math.floor(x)))
    y1 = max(0, int(math.floor(y)))
    x2 = min(width, int(math.ceil(x + w)))
    y2 = min(height, int(math.ceil(y + h)))
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2 - x1, y2 - y1


def _box_to_crop(
    box: tuple[int, int, int, int],
    origin: tuple[int, int],
    crop_width: int,
    crop_height: int,
) -> tuple[int, int, int, int] | None:
    x, y, w, h = box
    ox, oy = origin
    return _clip_box_to_shape((x - ox, y - oy, w, h), crop_width, crop_height)


def _point_to_box(
    point: np.ndarray,
    radius_px: float,
    width: int,
    height: int,
) -> tuple[int, int, int, int] | None:
    x, y = float(point[0]), float(point[1])
    return _clip_box_to_shape((x - radius_px, y - radius_px, 2 * radius_px, 2 * radius_px), width, height)


def _make_camera_feature_mask(
    shape: tuple[int, int],
    reticle_center_xy: tuple[float, float] | None = None,
    ignore_reticle: bool = True,
    reticle_center_fraction: float = 0.20,
    reticle_line_fraction: float = 0.025,
    ignore_boxes: list[tuple[int, int, int, int]] | None = None,
) -> np.ndarray:
    height, width = shape[:2]
    mask = np.full((height, width), 255, dtype=np.uint8)

    if ignore_reticle and reticle_center_xy is not None:
        cx, cy = reticle_center_xy
        line_half = max(1, int(round(min(width, height) * reticle_line_fraction / 2)))
        if 0 <= cy < height:
            y1 = max(0, int(round(cy)) - line_half)
            y2 = min(height, int(round(cy)) + line_half + 1)
            mask[y1:y2, :] = 0
        if 0 <= cx < width:
            x1 = max(0, int(round(cx)) - line_half)
            x2 = min(width, int(round(cx)) + line_half + 1)
            mask[:, x1:x2] = 0

        center_w = max(1, int(round(width * reticle_center_fraction)))
        center_h = max(1, int(round(height * reticle_center_fraction)))
        center_box = _clip_box_to_shape((cx - center_w / 2, cy - center_h / 2, center_w, center_h), width, height)
        if center_box is not None:
            x, y, w, h = center_box
            mask[y : y + h, x : x + w] = 0

    for box in ignore_boxes or []:
        clipped = _clip_box_to_shape(box, width, height)
        if clipped is None:
            continue
        x, y, w, h = clipped
        mask[y : y + h, x : x + w] = 0

    return mask


def _make_background_motion_mask(
    prev_gray: np.ndarray,
    curr_gray: np.ndarray,
    base_mask: np.ndarray | None = None,
    min_intensity: int = 12,
    max_intensity: int = 242,
    texture_percentile: float = 55.0,
) -> tuple[np.ndarray, dict[str, float]]:
    if base_mask is None:
        valid = np.ones(prev_gray.shape[:2], dtype=np.uint8) * 255
    else:
        valid = base_mask.copy()

    intensity_mask = (
        (prev_gray >= min_intensity)
        & (curr_gray >= min_intensity)
        & (prev_gray <= max_intensity)
        & (curr_gray <= max_intensity)
    ).astype(np.uint8) * 255
    kernel = np.ones((5, 5), np.uint8)
    intensity_mask = cv2.erode(intensity_mask, kernel, iterations=1)

    prev_sobel_x = cv2.Sobel(prev_gray, cv2.CV_32F, 1, 0, ksize=3)
    prev_sobel_y = cv2.Sobel(prev_gray, cv2.CV_32F, 0, 1, ksize=3)
    curr_sobel_x = cv2.Sobel(curr_gray, cv2.CV_32F, 1, 0, ksize=3)
    curr_sobel_y = cv2.Sobel(curr_gray, cv2.CV_32F, 0, 1, ksize=3)
    texture = np.maximum(
        cv2.magnitude(prev_sobel_x, prev_sobel_y),
        cv2.magnitude(curr_sobel_x, curr_sobel_y),
    )

    base_valid = (valid > 0) & (intensity_mask > 0)
    texture_values = texture[base_valid]
    if texture_values.size:
        texture_threshold = max(float(np.percentile(texture_values, texture_percentile)), 3.0)
    else:
        texture_threshold = 3.0
    texture_mask = (texture >= texture_threshold).astype(np.uint8) * 255

    background_mask = cv2.bitwise_and(valid, intensity_mask)
    background_mask = cv2.bitwise_and(background_mask, texture_mask)
    background_mask = cv2.morphologyEx(background_mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)

    metrics = {
        "camera_background_mask_coverage": float(np.count_nonzero(background_mask) / background_mask.size),
        "camera_background_texture_threshold": float(texture_threshold),
    }
    return background_mask, metrics


def _high_pass_for_matching(gray: np.ndarray) -> np.ndarray:
    gray_f = gray.astype(np.float32)
    blurred = cv2.GaussianBlur(gray_f, (0, 0), 5)
    high_pass = gray_f - blurred
    std = float(np.std(high_pass))
    if std > 1e-6:
        high_pass = high_pass / std
    return high_pass.astype(np.float32)


def _estimate_phase_translation(
    prev_gray: np.ndarray,
    curr_gray: np.ndarray,
    background_mask: np.ndarray,
    min_valid_fraction: float = 0.08,
    min_response: float = 0.015,
) -> tuple[np.ndarray | None, dict[str, Any]]:
    valid = background_mask > 0
    valid_fraction = float(np.count_nonzero(valid) / valid.size)
    if valid_fraction < min_valid_fraction:
        return None, {
            "camera_phase_dx_px": np.nan,
            "camera_phase_dy_px": np.nan,
            "camera_phase_response": np.nan,
            "camera_phase_valid_fraction": valid_fraction,
        }

    prev_hp = _high_pass_for_matching(prev_gray)
    curr_hp = _high_pass_for_matching(curr_gray)
    mask_f = valid.astype(np.float32)
    prev_mean = float(np.mean(prev_hp[valid]))
    curr_mean = float(np.mean(curr_hp[valid]))
    prev_prepared = (prev_hp - prev_mean) * mask_f
    curr_prepared = (curr_hp - curr_mean) * mask_f
    hanning = cv2.createHanningWindow((prev_gray.shape[1], prev_gray.shape[0]), cv2.CV_32F)
    try:
        (dx, dy), response = cv2.phaseCorrelate(prev_prepared, curr_prepared, hanning)
    except cv2.error:
        return None, {
            "camera_phase_dx_px": np.nan,
            "camera_phase_dy_px": np.nan,
            "camera_phase_response": np.nan,
            "camera_phase_valid_fraction": valid_fraction,
        }

    if not np.isfinite(dx) or not np.isfinite(dy) or not np.isfinite(response) or response < min_response:
        return None, {
            "camera_phase_dx_px": float(dx) if np.isfinite(dx) else np.nan,
            "camera_phase_dy_px": float(dy) if np.isfinite(dy) else np.nan,
            "camera_phase_response": float(response) if np.isfinite(response) else np.nan,
            "camera_phase_valid_fraction": valid_fraction,
        }

    affine = np.array([[1.0, 0.0, float(dx)], [0.0, 1.0, float(dy)]], dtype=np.float32)
    return affine, {
        "camera_phase_dx_px": float(dx),
        "camera_phase_dy_px": float(dy),
        "camera_phase_response": float(response),
        "camera_phase_valid_fraction": valid_fraction,
    }


def _estimate_block_translation(
    prev_gray: np.ndarray,
    curr_gray: np.ndarray,
    background_mask: np.ndarray,
    block_size: int = 96,
    stride: int = 64,
    search_radius: int = 96,
    min_mask_fraction: float = 0.65,
    min_score: float = 0.32,
) -> tuple[np.ndarray | None, dict[str, Any]]:
    height, width = prev_gray.shape[:2]
    prev_hp = _high_pass_for_matching(prev_gray)
    curr_hp = _high_pass_for_matching(curr_gray)
    translations: list[tuple[float, float, float]] = []

    if height < block_size or width < block_size:
        return None, {
            "camera_block_dx_px": np.nan,
            "camera_block_dy_px": np.nan,
            "camera_block_count": 0,
            "camera_block_score": np.nan,
        }

    for y in range(0, height - block_size + 1, stride):
        for x in range(0, width - block_size + 1, stride):
            block_mask = background_mask[y : y + block_size, x : x + block_size]
            if np.count_nonzero(block_mask) / block_mask.size < min_mask_fraction:
                continue

            prev_patch = prev_hp[y : y + block_size, x : x + block_size]
            if float(np.std(prev_patch[block_mask > 0])) < 4.0:
                continue

            sx1 = max(0, x - search_radius)
            sy1 = max(0, y - search_radius)
            sx2 = min(width, x + block_size + search_radius)
            sy2 = min(height, y + block_size + search_radius)
            search = curr_hp[sy1:sy2, sx1:sx2]
            if search.shape[0] < block_size or search.shape[1] < block_size:
                continue

            result = cv2.matchTemplate(search, prev_patch, cv2.TM_CCOEFF_NORMED)
            _, max_val, _, max_loc = cv2.minMaxLoc(result)
            if not np.isfinite(max_val) or max_val < min_score:
                continue
            match_x = sx1 + max_loc[0]
            match_y = sy1 + max_loc[1]
            translations.append((float(match_x - x), float(match_y - y), float(max_val)))

    if len(translations) < 4:
        return None, {
            "camera_block_dx_px": np.nan,
            "camera_block_dy_px": np.nan,
            "camera_block_count": len(translations),
            "camera_block_score": float(np.median([t[2] for t in translations])) if translations else np.nan,
        }

    arr = np.array(translations, dtype=float)
    median_xy = np.median(arr[:, :2], axis=0)
    residual = np.linalg.norm(arr[:, :2] - median_xy, axis=1)
    residual_median = float(np.median(residual))
    residual_mad = float(np.median(np.abs(residual - residual_median)))
    keep = residual <= residual_median + max(3.0, 3.0 * residual_mad)
    robust = arr[keep] if np.count_nonzero(keep) >= 4 else arr
    dx = float(np.median(robust[:, 0]))
    dy = float(np.median(robust[:, 1]))
    score = float(np.median(robust[:, 2]))
    affine = np.array([[1.0, 0.0, dx], [0.0, 1.0, dy]], dtype=np.float32)
    return affine, {
        "camera_block_dx_px": dx,
        "camera_block_dy_px": dy,
        "camera_block_count": int(len(robust)),
        "camera_block_score": score,
    }


def _estimate_global_motion(
    prev_gray: np.ndarray,
    curr_gray: np.ndarray,
    max_features: int = 2500,
    keep_percent: float = 0.25,
    ransac_reproj_threshold: float = 3.0,
    prev_feature_mask: np.ndarray | None = None,
    curr_feature_mask: np.ndarray | None = None,
    prefer_phase_translation: bool = True,
    prefer_block_translation: bool = True,
    prefer_dense_translation: bool = True,
) -> tuple[np.ndarray, dict[str, Any]]:
    affine = _identity_affine()
    metrics = {
        "camera_dx_px": 0.0,
        "camera_dy_px": 0.0,
        "camera_translation_px": 0.0,
        "camera_scale": 1.0,
        "camera_rotation_deg": 0.0,
        "camera_match_count": 0,
        "camera_inlier_ratio": 0.0,
        "camera_median_residual_px": np.nan,
        "camera_feature_mask_coverage": np.nan,
        "camera_dense_dx_px": np.nan,
        "camera_dense_dy_px": np.nan,
        "camera_dense_valid_fraction": np.nan,
        "camera_phase_dx_px": np.nan,
        "camera_phase_dy_px": np.nan,
        "camera_phase_response": np.nan,
        "camera_phase_valid_fraction": np.nan,
        "camera_block_dx_px": np.nan,
        "camera_block_dy_px": np.nan,
        "camera_block_count": 0,
        "camera_block_score": np.nan,
        "camera_background_mask_coverage": np.nan,
        "camera_background_texture_threshold": np.nan,
        "camera_motion_method": "none",
        "camera_motion_ok": False,
    }
    if prev_feature_mask is not None:
        metrics["camera_feature_mask_coverage"] = float(np.count_nonzero(prev_feature_mask) / prev_feature_mask.size)
    background_mask, background_metrics = _make_background_motion_mask(prev_gray, curr_gray, prev_feature_mask)
    metrics.update(background_metrics)

    phase_affine = None
    if prefer_phase_translation:
        phase_affine, phase_metrics = _estimate_phase_translation(prev_gray, curr_gray, background_mask)
        metrics.update(phase_metrics)
        if phase_affine is not None:
            tx = float(phase_affine[0, 2])
            ty = float(phase_affine[1, 2])
            metrics.update(
                {
                    "camera_dx_px": tx,
                    "camera_dy_px": ty,
                    "camera_translation_px": float(math.hypot(tx, ty)),
                    "camera_motion_method": "phase_correlation",
                    "camera_motion_ok": True,
                }
            )
            return phase_affine, metrics

    block_affine = None
    if prefer_block_translation:
        block_affine, block_metrics = _estimate_block_translation(prev_gray, curr_gray, background_mask)
        metrics.update(block_metrics)
        if block_affine is not None and metrics["camera_block_count"] >= 4:
            tx = float(block_affine[0, 2])
            ty = float(block_affine[1, 2])
            metrics.update(
                {
                    "camera_dx_px": tx,
                    "camera_dy_px": ty,
                    "camera_translation_px": float(math.hypot(tx, ty)),
                    "camera_motion_method": "block_match",
                    "camera_motion_ok": True,
                }
            )
            return block_affine, metrics

    dense_affine = None
    if prefer_dense_translation:
        flow = cv2.calcOpticalFlowFarneback(
            prev_gray,
            curr_gray,
            None,
            pyr_scale=0.5,
            levels=4,
            winsize=31,
            iterations=4,
            poly_n=7,
            poly_sigma=1.5,
            flags=0,
        )
        valid_mask = background_mask > 0
        flow_mag = np.linalg.norm(flow, axis=2)
        valid_mask &= np.isfinite(flow_mag)
        valid_mask &= flow_mag < max(prev_gray.shape[:2]) * 0.25
        if np.count_nonzero(valid_mask) >= max(500, int(prev_gray.size * 0.05)):
            dense_dx = float(np.median(flow[..., 0][valid_mask]))
            dense_dy = float(np.median(flow[..., 1][valid_mask]))
            dense_affine = np.array([[1.0, 0.0, dense_dx], [0.0, 1.0, dense_dy]], dtype=np.float32)
            metrics.update(
                {
                    "camera_dense_dx_px": dense_dx,
                    "camera_dense_dy_px": dense_dy,
                    "camera_dense_valid_fraction": float(np.count_nonzero(valid_mask) / valid_mask.size),
                }
            )

    orb = cv2.ORB_create(nfeatures=max_features, fastThreshold=7)
    prev_orb_mask = background_mask
    if curr_feature_mask is not None:
        curr_background_mask, _ = _make_background_motion_mask(curr_gray, prev_gray, curr_feature_mask)
    else:
        curr_background_mask, _ = _make_background_motion_mask(curr_gray, prev_gray, None)
    kp_prev, desc_prev = orb.detectAndCompute(prev_gray, prev_orb_mask)
    kp_curr, desc_curr = orb.detectAndCompute(curr_gray, curr_background_mask)
    if desc_prev is None or desc_curr is None or len(kp_prev) < 8 or len(kp_curr) < 8:
        if dense_affine is not None:
            tx = float(dense_affine[0, 2])
            ty = float(dense_affine[1, 2])
            metrics.update(
                {
                    "camera_dx_px": tx,
                    "camera_dy_px": ty,
                    "camera_translation_px": float(math.hypot(tx, ty)),
                    "camera_motion_method": "dense_flow",
                    "camera_motion_ok": True,
                }
            )
            return dense_affine, metrics
        return affine, metrics

    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = matcher.match(desc_prev, desc_curr)
    if len(matches) < 8:
        if dense_affine is not None:
            tx = float(dense_affine[0, 2])
            ty = float(dense_affine[1, 2])
            metrics.update(
                {
                    "camera_dx_px": tx,
                    "camera_dy_px": ty,
                    "camera_translation_px": float(math.hypot(tx, ty)),
                    "camera_motion_method": "dense_flow",
                    "camera_motion_ok": True,
                }
            )
            return dense_affine, metrics
        return affine, metrics

    matches = sorted(matches, key=lambda m: m.distance)
    keep = max(8, int(len(matches) * keep_percent))
    matches = matches[:keep]

    prev_pts = np.float32([kp_prev[m.queryIdx].pt for m in matches])
    curr_pts = np.float32([kp_curr[m.trainIdx].pt for m in matches])
    affine_est, inliers = cv2.estimateAffinePartial2D(
        prev_pts,
        curr_pts,
        method=cv2.RANSAC,
        ransacReprojThreshold=ransac_reproj_threshold,
        maxIters=2000,
        confidence=0.99,
    )
    if affine_est is None:
        metrics["camera_match_count"] = len(matches)
        if dense_affine is not None:
            tx = float(dense_affine[0, 2])
            ty = float(dense_affine[1, 2])
            metrics.update(
                {
                    "camera_dx_px": tx,
                    "camera_dy_px": ty,
                    "camera_translation_px": float(math.hypot(tx, ty)),
                    "camera_motion_method": "dense_flow",
                    "camera_motion_ok": True,
                }
            )
            return dense_affine, metrics
        return affine, metrics

    affine = affine_est.astype(np.float32)
    inlier_mask = inliers.ravel().astype(bool) if inliers is not None else np.ones(len(matches), dtype=bool)
    transformed = cv2.transform(prev_pts.reshape(-1, 1, 2), affine).reshape(-1, 2)
    residuals = np.linalg.norm(transformed - curr_pts, axis=1)
    residuals_inliers = residuals[inlier_mask] if np.any(inlier_mask) else residuals

    a, b, tx = affine[0]
    c, d, ty = affine[1]
    metrics.update(
        {
            "camera_dx_px": float(tx),
            "camera_dy_px": float(ty),
            "camera_translation_px": float(math.hypot(tx, ty)),
            "camera_scale": float(math.sqrt(max(a * a + c * c, 0.0))),
            "camera_rotation_deg": float(math.degrees(math.atan2(c, a))),
            "camera_match_count": int(len(matches)),
            "camera_inlier_ratio": float(np.mean(inlier_mask)),
            "camera_median_residual_px": float(np.median(residuals_inliers)),
            "camera_motion_method": "orb_features",
            "camera_motion_ok": True,
        }
    )
    if dense_affine is not None:
        dense_tx = float(dense_affine[0, 2])
        dense_ty = float(dense_affine[1, 2])
        feature_translation = float(metrics["camera_translation_px"])
        dense_translation = float(math.hypot(dense_tx, dense_ty))
        use_dense = (
            metrics["camera_inlier_ratio"] < 0.55
            or metrics["camera_median_residual_px"] > 2.5
            or dense_translation > max(2.0, feature_translation * 1.8)
        )
        if use_dense:
            affine = dense_affine
            metrics.update(
                {
                    "camera_dx_px": dense_tx,
                    "camera_dy_px": dense_ty,
                    "camera_translation_px": dense_translation,
                    "camera_scale": 1.0,
                    "camera_rotation_deg": 0.0,
                    "camera_motion_method": "dense_flow",
                }
            )
    return affine, metrics


def _detect_residual_blobs(
    prev_gray: np.ndarray,
    curr_gray: np.ndarray,
    affine: np.ndarray,
    min_blob_area: int = 4,
    max_blob_area_ratio: float = 0.01,
    diff_threshold: int | None = None,
) -> list[dict[str, Any]]:
    height, width = curr_gray.shape[:2]
    warped_prev = cv2.warpAffine(
        prev_gray,
        affine,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REPLICATE,
    )
    diff = cv2.absdiff(warped_prev, curr_gray)
    diff_blur = cv2.GaussianBlur(diff, (5, 5), 0)

    if diff_threshold is None:
        otsu_value, mask = cv2.threshold(diff_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        if otsu_value < 10:
            _, mask = cv2.threshold(diff_blur, 10, 255, cv2.THRESH_BINARY)
    else:
        _, mask = cv2.threshold(diff_blur, int(diff_threshold), 255, cv2.THRESH_BINARY)

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    max_blob_area = float(width * height * max_blob_area_ratio)
    candidates: list[dict[str, Any]] = []
    for contour in contours:
        area = float(cv2.contourArea(contour))
        if area < min_blob_area or area > max_blob_area:
            continue
        x, y, w, h = cv2.boundingRect(contour)
        moments = cv2.moments(contour)
        if moments["m00"]:
            cx = moments["m10"] / moments["m00"]
            cy = moments["m01"] / moments["m00"]
        else:
            cx = x + w / 2
            cy = y + h / 2
        patch = diff[y : y + h, x : x + w]
        score = area * (float(np.mean(patch)) + float(np.max(patch)))
        candidates.append(
            {
                "cx": float(cx),
                "cy": float(cy),
                "x": int(x),
                "y": int(y),
                "w": int(w),
                "h": int(h),
                "area": area,
                "score": float(score),
                "source": "residual",
            }
        )
    return sorted(candidates, key=lambda c: c["score"], reverse=True)


def _detect_bright_blobs(
    gray: np.ndarray,
    min_blob_area: int = 4,
    max_blob_area_ratio: float = 0.01,
    bright_percentile: float = 99.65,
    bright_threshold: int | None = None,
    ignore_boxes: list[tuple[int, int, int, int]] | None = None,
    min_fill_ratio: float = 0.18,
    max_aspect_ratio: float = 4.0,
) -> list[dict[str, Any]]:
    height, width = gray.shape[:2]
    working_mask = np.full((height, width), 255, dtype=np.uint8)
    for box in ignore_boxes or []:
        clipped = _clip_box_to_shape(box, width, height)
        if clipped is None:
            continue
        x, y, w, h = clipped
        working_mask[y : y + h, x : x + w] = 0

    valid_pixels = gray[working_mask > 0]
    if valid_pixels.size == 0:
        return []
    if bright_threshold is None:
        adaptive_threshold = float(np.percentile(valid_pixels, bright_percentile))
        contrast_threshold = float(np.mean(valid_pixels) + 2.5 * np.std(valid_pixels))
        threshold = int(np.clip(max(adaptive_threshold, contrast_threshold), 0, 255))
    else:
        threshold = int(bright_threshold)

    _, mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    mask = cv2.bitwise_and(mask, working_mask)
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    max_blob_area = float(width * height * max_blob_area_ratio)
    candidates: list[dict[str, Any]] = []
    for contour in contours:
        area = float(cv2.contourArea(contour))
        if area < min_blob_area or area > max_blob_area:
            continue
        x, y, w, h = cv2.boundingRect(contour)
        aspect_ratio = max(w / max(h, 1), h / max(w, 1))
        fill_ratio = area / max(float(w * h), 1.0)
        if aspect_ratio > max_aspect_ratio or fill_ratio < min_fill_ratio:
            continue
        moments = cv2.moments(contour)
        if moments["m00"]:
            cx = moments["m10"] / moments["m00"]
            cy = moments["m01"] / moments["m00"]
        else:
            cx = x + w / 2
            cy = y + h / 2
        patch = gray[y : y + h, x : x + w]
        patch_mask = mask[y : y + h, x : x + w]
        hot_pixels = patch[patch_mask > 0]
        mean_hot = float(np.mean(hot_pixels)) if hot_pixels.size else float(np.mean(patch))
        max_hot = float(np.max(patch))
        score = area * fill_ratio * (mean_hot + max_hot)
        candidates.append(
            {
                "cx": float(cx),
                "cy": float(cy),
                "x": int(x),
                "y": int(y),
                "w": int(w),
                "h": int(h),
                "area": area,
                "score": float(score),
                "source": "bright",
                "bright_threshold": threshold,
                "fill_ratio": float(fill_ratio),
                "aspect_ratio": float(aspect_ratio),
            }
        )
    return sorted(candidates, key=lambda c: c["score"], reverse=True)


def _choose_candidate(
    candidates: list[dict[str, Any]],
    predicted_center: np.ndarray | None,
    max_distance_px: float,
) -> tuple[dict[str, Any] | None, float]:
    if not candidates:
        return None, np.nan
    if predicted_center is None:
        return candidates[0], np.nan
    ranked = []
    for candidate in candidates:
        center = np.array([candidate["cx"], candidate["cy"]], dtype=float)
        distance = float(np.linalg.norm(center - predicted_center))
        ranked.append((distance, -candidate["score"], candidate))
    ranked.sort(key=lambda item: (item[0], item[1]))
    distance, _, candidate = ranked[0]
    if distance <= max_distance_px:
        return candidate, distance
    return None, distance


def _median_or_nan(values: pd.Series) -> float:
    values = pd.to_numeric(values, errors="coerce").dropna()
    if values.empty:
        return float("nan")
    return float(values.median())


def _ratio(after: float, before: float) -> float:
    if not np.isfinite(after) or not np.isfinite(before):
        return float("nan")
    return float(after / max(abs(before), 1e-6))


def _classify_disappearance(
    track: pd.DataFrame,
    missing_start_pos: int,
    missing_end_pos: int,
    edge_margin_px: float,
    window: int = 8,
) -> dict[str, Any]:
    before = track.iloc[max(0, missing_start_pos - window) : missing_start_pos].copy()
    after = track.iloc[missing_start_pos : min(len(track), missing_start_pos + window)].copy()
    detected_before = before[before["detected"]]

    if detected_before.empty:
        return {}

    last = detected_before.iloc[-1]
    early = detected_before.iloc[: max(1, len(detected_before) // 2)]
    late = detected_before.iloc[max(0, len(detected_before) - max(2, len(detected_before) // 2)) :]

    camera_before = _median_or_nan(before["camera_speed_px_s"])
    camera_after = _median_or_nan(after["camera_speed_px_s"])
    camera_change_ratio = _ratio(camera_after, camera_before)
    camera_after_dx_px = float(pd.to_numeric(after["camera_dx_px"], errors="coerce").fillna(0).sum())
    camera_after_dy_px = float(pd.to_numeric(after["camera_dy_px"], errors="coerce").fillna(0).sum())
    camera_after_translation_px = float(math.hypot(camera_after_dx_px, camera_after_dy_px))

    object_speed_early = _median_or_nan(early["uap_residual_speed_px_s"])
    object_speed_late = _median_or_nan(late["uap_residual_speed_px_s"])
    object_speed_ratio = _ratio(object_speed_late, object_speed_early)
    object_accel_late = _median_or_nan(late["uap_residual_accel_px_s2"])
    object_vx_late = _median_or_nan(late["uap_residual_vx_px_s"]) if "uap_residual_vx_px_s" in late else np.nan
    object_vy_late = _median_or_nan(late["uap_residual_vy_px_s"]) if "uap_residual_vy_px_s" in late else np.nan

    decision_lookback = max(window * 5, 36)
    decision_start = max(0, missing_start_pos - decision_lookback)
    decision_window = track.iloc[decision_start:missing_start_pos].copy()
    decision_window = decision_window[decision_window["detected"]].copy()
    if len(decision_window) > 6:
        decision_window = decision_window.iloc[:-2].copy()
    speed_series_raw = pd.to_numeric(decision_window["uap_residual_speed_px_s"], errors="coerce")
    accel_series_raw = pd.to_numeric(decision_window["uap_residual_accel_px_s2"], errors="coerce")
    speed_series = speed_series_raw.rolling(3, min_periods=1).median()
    accel_series = accel_series_raw.rolling(3, min_periods=1).median()
    baseline_count = max(5, min(len(decision_window) // 2, 18))
    baseline_speed = speed_series.iloc[:baseline_count].dropna()
    baseline_accel = accel_series.iloc[:baseline_count].dropna()
    speed_floor = float(baseline_speed.median()) if not baseline_speed.empty else 0.0
    speed_mad = float((baseline_speed - speed_floor).abs().median()) if not baseline_speed.empty else 0.0
    accel_floor = float(baseline_accel.abs().median()) if not baseline_accel.empty else 0.0
    accel_mad = float((baseline_accel.abs() - accel_floor).abs().median()) if not baseline_accel.empty else 0.0
    speed_threshold = max(speed_floor * 2.25, speed_floor + 5 * speed_mad, 300.0)
    accel_threshold = max(accel_floor + 6 * accel_mad, 1800.0)
    acceleration_mask = (speed_series >= speed_threshold) & (accel_series.abs() >= accel_threshold)
    sustained_speed_mask = (speed_series >= speed_threshold) & (speed_series.shift(-1) >= speed_threshold * 0.75)
    acceleration_candidates = decision_window[acceleration_mask]
    if acceleration_candidates.empty:
        acceleration_candidates = decision_window[sustained_speed_mask]
    if acceleration_candidates.empty:
        decision = last
        decision_reason = "fallback_last_seen"
    else:
        decision = acceleration_candidates.iloc[0]
        decision_reason = "first_sustained_acceleration"

    near_edge = bool(float(last.get("distance_to_edge_px", np.inf)) <= edge_margin_px)
    camera_stopped = bool(np.isfinite(camera_before) and np.isfinite(camera_after) and camera_before > 10 and camera_after < camera_before * 0.45)
    camera_moved = bool(np.isfinite(camera_before) and np.isfinite(camera_after) and camera_after > max(10, camera_before * 1.7))
    object_accelerated = bool(
        (np.isfinite(object_speed_ratio) and object_speed_ratio > 1.6 and object_speed_late > 8)
        or (np.isfinite(object_accel_late) and object_accel_late > 25)
    )
    object_decelerated = bool(
        (np.isfinite(object_speed_ratio) and object_speed_ratio < 0.55 and object_speed_early > 8)
        or (np.isfinite(object_accel_late) and object_accel_late < -25)
    )

    if camera_stopped and near_edge:
        classification = "camera_stopped_tracking"
        confidence = 0.78
        reason = "The global camera motion dropped around the disappearance and the target was near the frame edge."
    elif camera_moved and not object_accelerated:
        classification = "camera_moved"
        confidence = 0.72
        reason = "The global scene motion increased at the disappearance, without strong evidence of target residual acceleration."
    elif object_accelerated and near_edge:
        classification = "uap_accelerated_out_of_scene"
        confidence = 0.76
        reason = "The target residual speed increased and the last detection was near the frame edge."
    elif object_accelerated:
        classification = "uap_accelerated"
        confidence = 0.64
        reason = "The target residual speed/acceleration increased after compensating for camera motion."
    elif object_decelerated:
        classification = "uap_decelerated_or_lost_contrast"
        confidence = 0.58
        reason = "Residual speed dropped before the disappearance; this may indicate deceleration or loss of contrast/detection."
    elif camera_moved:
        classification = "camera_moved"
        confidence = 0.62
        reason = "There is a relevant change in global camera motion near the disappearance."
    else:
        classification = "undetermined"
        confidence = 0.35
        reason = "The camera-motion and target-residual evidence did not separate a dominant cause."

    return {
        "missing_start_frame": int(track.iloc[missing_start_pos]["frame_index"]),
        "missing_start_time_s": float(track.iloc[missing_start_pos]["time_s"]),
        "decision_frame": int(decision["frame_index"]),
        "decision_time_s": float(decision["time_s"]),
        "decision_reason": decision_reason,
        "decision_lookback_frames": int(decision_lookback),
        "decision_speed_threshold_px_s": speed_threshold,
        "decision_accel_threshold_px_s2": accel_threshold,
        "last_seen_frame": int(last["frame_index"]),
        "last_seen_time_s": float(last["time_s"]),
        "missing_frames": int(missing_end_pos - missing_start_pos),
        "classification": classification,
        "confidence": confidence,
        "reason": reason,
        "camera_speed_before_px_s": camera_before,
        "camera_speed_after_px_s": camera_after,
        "camera_change_ratio": camera_change_ratio,
        "camera_after_cumulative_dx_px": camera_after_dx_px,
        "camera_after_cumulative_dy_px": camera_after_dy_px,
        "camera_after_cumulative_translation_px": camera_after_translation_px,
        "uap_speed_early_px_s": object_speed_early,
        "uap_speed_late_px_s": object_speed_late,
        "uap_speed_ratio": object_speed_ratio,
        "uap_residual_vx_late_px_s": object_vx_late,
        "uap_residual_vy_late_px_s": object_vy_late,
        "uap_accel_late_px_s2": object_accel_late,
        "last_distance_to_edge_px": float(last.get("distance_to_edge_px", np.nan)),
    }


def _find_disappearance_events(
    track: pd.DataFrame,
    disappearance_patience: int,
    edge_margin_px: float,
) -> pd.DataFrame:
    if track.empty or "detected" not in track:
        return pd.DataFrame()
    detected = track["detected"].fillna(False).to_numpy(dtype=bool)
    events = []
    pos = 1
    while pos < len(detected):
        if detected[pos - 1] and not detected[pos]:
            start = pos
            while pos < len(detected) and not detected[pos]:
                pos += 1
            end = pos
            if end - start >= disappearance_patience:
                event = _classify_disappearance(track, start, end, edge_margin_px=edge_margin_px)
                if event:
                    events.append(event)
        else:
            pos += 1
    return pd.DataFrame(events)


def analyze_uap_video(
    video_path: str | Path,
    output_dir: str | Path | None = None,
    roi: tuple[int, int, int, int] | None = None,
    object_bbox: tuple[int, int, int, int] | None = None,
    camera_motion_ignore_boxes: list[tuple[int, int, int, int]] | None = None,
    ignore_reticle: bool = True,
    reticle_center_fraction: float = 0.20,
    reticle_line_fraction: float = 0.025,
    target_mask_radius_px: float | None = None,
    frame_step: int = 1,
    image_ext: str = "png",
    disappearance_patience: int = 5,
    min_blob_area: int = 4,
    max_blob_area_ratio: float = 0.01,
    diff_threshold: int | None = None,
    target_detection_mode: str = "combined",
    bright_percentile: float = 99.65,
    bright_threshold: int | None = None,
    max_tracking_jump_px: float | None = None,
    verbose: bool = True,
) -> dict[str, Any]:
    """Extract video frames and classify disappearances of a possible UAP.

    Main parameters
    ---------------------
    video_path:
        Path to the video file.
    output_dir:
        Output folder. If omitted, a folder is created next to the video.
    roi:
        Region of interest in the full frame, as (x, y, width, height).
        Use this to limit analysis to the area where the UAP appears.
    object_bbox:
        Approximate initial UAP box in the full frame, as (x, y, width, height).
        This helps a lot when there are other moving objects in the video.
    camera_motion_ignore_boxes:
        Full-frame boxes to exclude from camera-motion estimation, such as HUD text or fixed overlays.
    ignore_reticle:
        Exclude the central reticle/crosshair area from camera-motion feature matching. This should stay True for targeting-pod, drone, or fighter footage where the sight is locked to the UAP.
    reticle_center_fraction:
        Fraction of the analyzed width/height to mask around the reticle center for camera-motion estimation.
    reticle_line_fraction:
        Relative thickness of the horizontal/vertical crosshair bands excluded from camera-motion estimation.
    target_mask_radius_px:
        Radius around the last UAP position to exclude from camera-motion estimation. If omitted, a resolution-scaled value is used.
    target_detection_mode:
        `combined` uses both residual motion and bright thermal hotspot detection. Use `bright` for thermal footage with a compact hot target, or `residual` to use only frame differencing.
    bright_percentile:
        Adaptive percentile threshold for thermal hotspot detection when `bright_threshold` is omitted.
    bright_threshold:
        Fixed grayscale threshold for thermal hotspot detection. Leave None for adaptive thresholding.
    frame_step:
        Process/export 1 out of every N frames. Keep 1 to export all frames.

    Returns
    -------
    dict with metadata, frame/track/event DataFrames, and the frame output folder.
    """
    video_path = Path(video_path).expanduser().resolve()
    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")
    if frame_step < 1:
        raise ValueError("frame_step must be >= 1")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = _safe_fps(cap)
    frame_count_nominal = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    duration_s = frame_count_nominal / fps if fps else np.nan

    roi = _validate_box(roi, width, height, "roi")
    object_bbox = _validate_box(object_bbox, width, height, "object_bbox")
    camera_motion_ignore_boxes = _validate_boxes(camera_motion_ignore_boxes, width, height, "camera_motion_ignore_boxes")
    if not 0 <= reticle_center_fraction <= 1:
        raise ValueError("reticle_center_fraction must be between 0 and 1")
    if not 0 <= reticle_line_fraction <= 1:
        raise ValueError("reticle_line_fraction must be between 0 and 1")
    target_detection_mode = target_detection_mode.lower().strip()
    if target_detection_mode not in {"combined", "residual", "bright"}:
        raise ValueError("target_detection_mode must be 'combined', 'residual', or 'bright'")
    if not 0 < bright_percentile <= 100:
        raise ValueError("bright_percentile must be between 0 and 100")

    if output_dir is None:
        output_dir = video_path.parent / f"{video_path.stem}_uap_analysis"
    output_dir = Path(output_dir).expanduser().resolve()
    frames_dir = output_dir / "frames"
    frames_dir.mkdir(parents=True, exist_ok=True)

    roi_width = roi[2] if roi else width
    roi_height = roi[3] if roi else height
    edge_margin_px = 0.08 * math.hypot(roi_width, roi_height)
    if max_tracking_jump_px is None:
        max_tracking_jump_px = max(12.0, 0.06 * math.hypot(roi_width, roi_height))
    if target_mask_radius_px is None:
        target_mask_radius_px = max(16.0, 0.04 * math.hypot(roi_width, roi_height))

    frame_rows: list[dict[str, Any]] = []
    track_rows: list[dict[str, Any]] = []

    prev_gray: np.ndarray | None = None
    prev_frame_index: int | None = None
    prev_time_s: float | None = None
    last_center: np.ndarray | None = None
    residual_velocity = np.array([0.0, 0.0], dtype=float)
    last_residual_speed = np.nan
    started_tracking = False

    read_index = 0
    saved_count = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame_index = read_index
        read_index += 1
        if frame_index % frame_step != 0:
            continue

        time_s = frame_index / fps
        frame_path = frames_dir / f"frame_{frame_index:06d}.{image_ext}"
        if not cv2.imwrite(str(frame_path), frame):
            raise IOError(f"Failed to save frame: {frame_path}")

        frame_rows.append(
            {
                "frame_index": frame_index,
                "time_s": time_s,
                "path": str(frame_path),
                "width": width,
                "height": height,
            }
        )
        saved_count += 1

        cropped, origin = _crop(frame, roi)
        gray = _to_gray(cropped)

        if not started_tracking and object_bbox is not None:
            bx, by, bw, bh = object_bbox
            ox, oy = origin
            initial_center = np.array([bx + bw / 2 - ox, by + bh / 2 - oy], dtype=float)
            if 0 <= initial_center[0] < roi_width and 0 <= initial_center[1] < roi_height:
                last_center = initial_center
                started_tracking = True

        if prev_gray is None:
            prev_gray = gray
            prev_frame_index = frame_index
            prev_time_s = time_s
            continue

        dt = max(time_s - float(prev_time_s), 1.0 / fps)

        camera_ignore_boxes = []
        for ignore_box in camera_motion_ignore_boxes:
            crop_box = _box_to_crop(ignore_box, origin, roi_width, roi_height)
            if crop_box is not None:
                camera_ignore_boxes.append(crop_box)
        if object_bbox is not None:
            crop_box = _box_to_crop(object_bbox, origin, roi_width, roi_height)
            if crop_box is not None:
                camera_ignore_boxes.append(crop_box)
        if last_center is not None:
            center_box = _point_to_box(last_center, float(target_mask_radius_px), roi_width, roi_height)
            if center_box is not None:
                camera_ignore_boxes.append(center_box)

        reticle_center_xy = (width / 2 - origin[0], height / 2 - origin[1])
        camera_feature_mask = _make_camera_feature_mask(
            gray.shape,
            reticle_center_xy=reticle_center_xy,
            ignore_reticle=ignore_reticle,
            reticle_center_fraction=reticle_center_fraction,
            reticle_line_fraction=reticle_line_fraction,
            ignore_boxes=camera_ignore_boxes,
        )
        affine, camera_metrics = _estimate_global_motion(
            prev_gray,
            gray,
            prev_feature_mask=camera_feature_mask,
            curr_feature_mask=camera_feature_mask,
        )
        camera_metrics["camera_ignore_box_count"] = len(camera_ignore_boxes)
        residual_candidates: list[dict[str, Any]] = []
        bright_candidates: list[dict[str, Any]] = []
        if target_detection_mode in {"combined", "residual"}:
            residual_candidates = _detect_residual_blobs(
                prev_gray,
                gray,
                affine,
                min_blob_area=min_blob_area,
                max_blob_area_ratio=max_blob_area_ratio,
                diff_threshold=diff_threshold,
            )
        if target_detection_mode in {"combined", "bright"}:
            target_ignore_boxes = []
            for ignore_box in camera_motion_ignore_boxes:
                crop_box = _box_to_crop(ignore_box, origin, roi_width, roi_height)
                if crop_box is not None:
                    target_ignore_boxes.append(crop_box)
            bright_candidates = _detect_bright_blobs(
                gray,
                min_blob_area=min_blob_area,
                max_blob_area_ratio=max_blob_area_ratio,
                bright_percentile=bright_percentile,
                bright_threshold=bright_threshold,
                ignore_boxes=target_ignore_boxes,
            )
        candidates = sorted(residual_candidates + bright_candidates, key=lambda c: c["score"], reverse=True)

        camera_pred = None
        predicted_center = None
        if last_center is not None:
            camera_pred = _transform_point(last_center, affine)
            predicted_center = camera_pred + residual_velocity * dt

        chosen, nearest_distance = _choose_candidate(candidates, predicted_center, float(max_tracking_jump_px))
        if chosen is not None:
            measured_center = np.array([chosen["cx"], chosen["cy"]], dtype=float)
            if camera_pred is not None:
                residual_displacement = measured_center - camera_pred
                residual_dx = float(residual_displacement[0])
                residual_dy = float(residual_displacement[1])
                residual_speed = float(np.linalg.norm(residual_displacement) / dt)
                residual_velocity = residual_displacement / dt
                residual_vx = float(residual_velocity[0])
                residual_vy = float(residual_velocity[1])
                residual_accel = (
                    float((residual_speed - last_residual_speed) / dt)
                    if np.isfinite(last_residual_speed)
                    else np.nan
                )
            else:
                residual_displacement = np.array([np.nan, np.nan])
                residual_dx = np.nan
                residual_dy = np.nan
                residual_speed = np.nan
                residual_vx = np.nan
                residual_vy = np.nan
                residual_accel = np.nan
                residual_velocity = np.array([0.0, 0.0], dtype=float)

            last_center = measured_center
            last_residual_speed = residual_speed
            started_tracking = True
            detected = True
            distance_to_edge = _edge_distance(measured_center, roi_width, roi_height)
            uap_x_px = float(measured_center[0] + origin[0])
            uap_y_px = float(measured_center[1] + origin[1])
            predicted_x_px = float(predicted_center[0] + origin[0]) if predicted_center is not None else np.nan
            predicted_y_px = float(predicted_center[1] + origin[1]) if predicted_center is not None else np.nan
            blob_area = float(chosen["area"])
            blob_score = float(chosen["score"])
            candidate_source = chosen.get("source", "unknown")
            bright_candidate_threshold = chosen.get("bright_threshold", np.nan)
        else:
            detected = False
            if predicted_center is not None:
                last_center = predicted_center
            distance_to_edge = _edge_distance(last_center, roi_width, roi_height) if last_center is not None else np.nan
            residual_speed = np.nan
            residual_dx = np.nan
            residual_dy = np.nan
            residual_vx = np.nan
            residual_vy = np.nan
            residual_accel = np.nan
            uap_x_px = np.nan
            uap_y_px = np.nan
            predicted_x_px = float(predicted_center[0] + origin[0]) if predicted_center is not None else np.nan
            predicted_y_px = float(predicted_center[1] + origin[1]) if predicted_center is not None else np.nan
            blob_area = np.nan
            blob_score = np.nan
            candidate_source = "none"
            bright_candidate_threshold = np.nan

        row = {
            "frame_index": frame_index,
            "prev_frame_index": int(prev_frame_index),
            "time_s": time_s,
            "dt_s": dt,
            "detected": detected,
            "candidate_count": len(candidates),
            "residual_candidate_count": len(residual_candidates),
            "bright_candidate_count": len(bright_candidates),
            "nearest_candidate_distance_px": nearest_distance,
            "candidate_source": candidate_source,
            "bright_candidate_threshold": bright_candidate_threshold,
            "uap_x_px": uap_x_px,
            "uap_y_px": uap_y_px,
            "predicted_uap_x_px": predicted_x_px,
            "predicted_uap_y_px": predicted_y_px,
            "uap_blob_area_px": blob_area,
            "uap_blob_score": blob_score,
            "uap_residual_speed_px_s": residual_speed,
            "uap_residual_dx_px": residual_dx,
            "uap_residual_dy_px": residual_dy,
            "uap_residual_vx_px_s": residual_vx,
            "uap_residual_vy_px_s": residual_vy,
            "uap_residual_accel_px_s2": residual_accel,
            "distance_to_edge_px": distance_to_edge,
            **camera_metrics,
        }
        row["camera_speed_px_s"] = row["camera_translation_px"] / dt
        row["camera_pan_dx_px"] = -row["camera_dx_px"]
        row["camera_pan_dy_px"] = -row["camera_dy_px"]
        row["camera_pan_speed_px_s"] = row["camera_speed_px_s"]
        track_rows.append(row)

        prev_gray = gray
        prev_frame_index = frame_index
        prev_time_s = time_s

    cap.release()

    frames = pd.DataFrame(frame_rows)
    track = pd.DataFrame(track_rows)
    if not track.empty:
        track["camera_cumulative_dx_px"] = track["camera_dx_px"].fillna(0).cumsum()
        track["camera_cumulative_dy_px"] = track["camera_dy_px"].fillna(0).cumsum()
        track["camera_pan_cumulative_dx_px"] = track["camera_pan_dx_px"].fillna(0).cumsum()
        track["camera_pan_cumulative_dy_px"] = track["camera_pan_dy_px"].fillna(0).cumsum()
        track["camera_cumulative_translation_px"] = np.hypot(
            track["camera_cumulative_dx_px"],
            track["camera_cumulative_dy_px"],
        )
        track["uap_residual_cumulative_dx_px"] = track["uap_residual_dx_px"].fillna(0).cumsum()
        track["uap_residual_cumulative_dy_px"] = track["uap_residual_dy_px"].fillna(0).cumsum()
    events = _find_disappearance_events(track, disappearance_patience, edge_margin_px=edge_margin_px)

    metadata = {
        "video_path": str(video_path),
        "fps": fps,
        "nominal_frame_count": frame_count_nominal,
        "processed_frame_count": len(frames),
        "width": width,
        "height": height,
        "duration_s": duration_s,
        "frame_step": frame_step,
        "frames_dir": str(frames_dir),
        "roi": roi,
        "object_bbox": object_bbox,
        "camera_motion_ignore_boxes": camera_motion_ignore_boxes,
        "ignore_reticle": ignore_reticle,
        "reticle_center_fraction": reticle_center_fraction,
        "reticle_line_fraction": reticle_line_fraction,
        "target_mask_radius_px": target_mask_radius_px,
        "target_detection_mode": target_detection_mode,
        "bright_percentile": bright_percentile,
        "bright_threshold": bright_threshold,
        "edge_margin_px": edge_margin_px,
        "max_tracking_jump_px": max_tracking_jump_px,
    }

    if verbose:
        print(f"Detected FPS: {fps:.3f}")
        print(f"Exported frames: {len(frames)} to {frames_dir}")
        if events.empty:
            print("No persistent disappearance was detected with the current parameters.")
        else:
            display_cols = [
                "last_seen_frame",
                "decision_frame",
                "missing_start_frame",
                "classification",
                "confidence",
                "decision_reason",
                "reason",
            ]
            display(events[display_cols])

    return {
        "metadata": metadata,
        "frames": frames,
        "track": track,
        "events": events,
        "frames_dir": frames_dir,
    }

def plot_diagnostics(result: dict[str, Any]):
    """Plot camera motion, UAP residual speed, and residual acceleration."""
    track = result["track"]
    events = result["events"]
    if track.empty:
        raise ValueError("There is no track data to plot.")

    fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
    camera_speed = pd.to_numeric(track["camera_speed_px_s"], errors="coerce")
    camera_speed_smooth = camera_speed.rolling(5, min_periods=1, center=True).median()
    axes[0].plot(track["time_s"], camera_speed, color="tab:blue", alpha=0.25, label="camera raw")
    axes[0].plot(track["time_s"], camera_speed_smooth, color="tab:blue", label="camera smoothed")
    axes[0].set_ylabel("Camera\n(px/s)")
    axes[0].grid(True, alpha=0.25)

    detected = track[track["detected"]]
    axes[1].plot(
        detected["time_s"],
        detected["uap_residual_speed_px_s"],
        color="tab:orange",
        marker="o",
        ms=3,
        label="UAP residual",
    )
    axes[1].set_ylabel("UAP residual\n(px/s)")
    axes[1].grid(True, alpha=0.25)

    axes[2].plot(
        detected["time_s"],
        detected["uap_residual_accel_px_s2"],
        color="tab:green",
        marker="o",
        ms=3,
        label="residual acceleration",
    )
    axes[2].axhline(0, color="black", lw=0.8, alpha=0.4)
    axes[2].set_ylabel("Acceleration\n(px/s2)")
    axes[2].set_xlabel("Time (s)")
    axes[2].grid(True, alpha=0.25)

    if not events.empty:
        for _, event in events.iterrows():
            decision_time = event.get("decision_time_s", event["last_seen_time_s"])
            for ax in axes:
                ax.axvline(decision_time, color="crimson", ls="-", alpha=0.9)
                ax.axvline(event["last_seen_time_s"], color="crimson", ls=":", alpha=0.55)
                ax.axvline(event["missing_start_time_s"], color="gray", ls="--", alpha=0.45)
            axes[0].annotate(
                event["classification"],
                xy=(decision_time, axes[0].get_ylim()[1]),
                xytext=(5, -15),
                textcoords="offset points",
                color="crimson",
                fontsize=9,
                va="top",
            )

    fig.tight_layout()
    return fig


def plot_axis_diagnostics(result: dict[str, Any]):
    """Plot camera X/Y motion and UAP X/Y position to diagnose axis-specific behavior."""
    track = result["track"]
    events = result["events"]
    if track.empty:
        raise ValueError("There is no track data to plot.")
    if "camera_cumulative_dx_px" not in track or "camera_cumulative_dy_px" not in track:
        track = track.copy()
        track["camera_cumulative_dx_px"] = track["camera_dx_px"].fillna(0).cumsum()
        track["camera_cumulative_dy_px"] = track["camera_dy_px"].fillna(0).cumsum()
    if "camera_pan_dx_px" not in track or "camera_pan_dy_px" not in track:
        track = track.copy()
        track["camera_pan_dx_px"] = -track["camera_dx_px"]
        track["camera_pan_dy_px"] = -track["camera_dy_px"]
    if "uap_residual_vx_px_s" not in track or "uap_residual_vy_px_s" not in track:
        track = track.copy()
        track["uap_residual_vx_px_s"] = np.nan
        track["uap_residual_vy_px_s"] = np.nan

    detected = track[track["detected"]]
    predicted_when_detected = detected.dropna(subset=["predicted_uap_x_px", "predicted_uap_y_px"])
    fig, axes = plt.subplots(7, 1, figsize=(13, 15), sharex=True)

    camera_dx = pd.to_numeric(track["camera_dx_px"], errors="coerce")
    camera_dy = pd.to_numeric(track["camera_dy_px"], errors="coerce")
    camera_pan_dx = pd.to_numeric(track["camera_pan_dx_px"], errors="coerce")
    camera_dx_smooth = camera_dx.rolling(5, min_periods=1, center=True).median()
    camera_dy_smooth = camera_dy.rolling(5, min_periods=1, center=True).median()
    camera_pan_dx_smooth = camera_pan_dx.rolling(5, min_periods=1, center=True).median()

    axes[0].plot(track["time_s"], camera_dx, color="tab:blue", alpha=0.25, label="camera dx raw")
    axes[0].plot(track["time_s"], camera_dx_smooth, color="tab:blue", label="camera dx smoothed")
    axes[0].plot(track["time_s"], camera_pan_dx_smooth, color="tab:purple", ls="--", label="camera pan dx smoothed")
    axes[0].axhline(0, color="black", lw=0.8, alpha=0.35)
    axes[0].set_ylabel("Camera dx\n(px/step)")
    axes[0].grid(True, alpha=0.25)

    axes[1].plot(track["time_s"], camera_dy, color="tab:cyan", alpha=0.25, label="camera dy raw")
    axes[1].plot(track["time_s"], camera_dy_smooth, color="tab:cyan", label="camera dy smoothed")
    axes[1].axhline(0, color="black", lw=0.8, alpha=0.35)
    axes[1].set_ylabel("Camera dy\n(px/step)")
    axes[1].grid(True, alpha=0.25)

    axes[2].plot(
        track["time_s"],
        track["camera_cumulative_dx_px"],
        color="tab:blue",
        label="cumulative camera dx",
    )
    axes[2].plot(
        track["time_s"],
        track["camera_cumulative_dy_px"],
        color="tab:cyan",
        label="cumulative camera dy",
    )
    axes[2].axhline(0, color="black", lw=0.8, alpha=0.35)
    axes[2].set_ylabel("Camera cumulative\n(px)")
    axes[2].grid(True, alpha=0.25)

    axes[3].plot(
        detected["time_s"],
        detected["uap_x_px"],
        color="tab:orange",
        marker="o",
        ms=3,
        label="detected UAP x",
    )
    axes[3].plot(
        predicted_when_detected["time_s"],
        predicted_when_detected["predicted_uap_x_px"],
        color="tab:orange",
        alpha=0.35,
        ls="--",
        label="predicted UAP x",
    )
    axes[3].set_ylabel("UAP x\n(px)")
    axes[3].grid(True, alpha=0.25)

    axes[4].plot(
        detected["time_s"],
        detected["uap_y_px"],
        color="tab:green",
        marker="o",
        ms=3,
        label="detected UAP y",
    )
    axes[4].plot(
        predicted_when_detected["time_s"],
        predicted_when_detected["predicted_uap_y_px"],
        color="tab:green",
        alpha=0.35,
        ls="--",
        label="predicted UAP y",
    )
    axes[4].set_ylabel("UAP y\n(px, image coords)")
    axes[4].invert_yaxis()
    axes[4].grid(True, alpha=0.25)

    axes[5].plot(
        detected["time_s"],
        detected["uap_residual_vx_px_s"],
        color="tab:orange",
        marker="o",
        ms=3,
        label="UAP residual vx",
    )
    axes[5].plot(
        detected["time_s"],
        detected["uap_residual_vy_px_s"],
        color="tab:green",
        marker="o",
        ms=3,
        label="UAP residual vy",
    )
    axes[5].axhline(0, color="black", lw=0.8, alpha=0.35)
    axes[5].set_ylabel("UAP residual\nvelocity (px/s)")
    axes[5].grid(True, alpha=0.25)

    axes[6].plot(
        detected["time_s"],
        detected["uap_residual_speed_px_s"],
        color="tab:red",
        marker="o",
        ms=3,
        label="UAP residual speed",
    )
    axes[6].set_ylabel("Residual speed\n(px/s)")
    axes[6].set_xlabel("Time (s)")
    axes[6].grid(True, alpha=0.25)

    for ax in axes:
        ax.legend(loc="upper right", fontsize=8)

    if not events.empty:
        for _, event in events.iterrows():
            decision_time = event.get("decision_time_s", event["last_seen_time_s"])
            for ax in axes:
                ax.axvline(decision_time, color="crimson", ls="-", alpha=0.9)
                ax.axvline(event["last_seen_time_s"], color="crimson", ls=":", alpha=0.55)
                ax.axvline(event["missing_start_time_s"], color="gray", ls="--", alpha=0.45)
            axes[0].annotate(
                event["classification"],
                xy=(decision_time, axes[0].get_ylim()[1]),
                xytext=(5, -15),
                textcoords="offset points",
                color="crimson",
                fontsize=9,
                va="top",
            )

    fig.tight_layout()
    return fig

## How to Use

The simplest call uses only the video path. For videos with many moving objects, also provide a `roi` and/or an approximate `object_bbox` in the first frame where the UAP appears.

- `roi=(x, y, width, height)` limits analysis to part of the frame.
- `object_bbox=(x, y, width, height)` initializes tracking on the correct target.
- `ignore_reticle=True` excludes the central weapon/sensor reticle from camera-motion estimation.
- `camera_motion_ignore_boxes=[...]` excludes HUD text, fixed overlays, or other non-background areas from camera-motion estimation.
- `target_detection_mode="combined"` uses both residual motion and compact thermal hotspot detection. This is usually better for weapon-sensor footage.
- `disappearance_patience` defines how many frames without detection count as a real disappearance.

In [ ]:
result = analyze_uap_video(
    "/content/hyperaccel1.mp4",
    output_dir="/content/uap_analysis_output",
    roi=None,                 # example: (100, 80, 900, 500)
    object_bbox=None,         # example: (460, 220, 18, 18)
    camera_motion_ignore_boxes=None,
    ignore_reticle=True,
    reticle_center_fraction=0.20,
    reticle_line_fraction=0.025,
    target_detection_mode="combined",
    bright_percentile=99.65,
    frame_step=1,
    disappearance_patience=5,
)

result["metadata"]
result["events"]
plot_diagnostics(result);
plot_axis_diagnostics(result);


## Interpreting the Results

The `result["events"]` DataFrame summarizes each persistent disappearance. Possible classifications are:

- `uap_accelerated_out_of_scene`: the target accelerated after camera compensation and was near the frame edge.
- `uap_accelerated`: residual speed/acceleration increased, but without clear evidence of leaving through the edge.
- `uap_decelerated_or_lost_contrast`: residual target speed dropped before disappearance; this may be deceleration or contrast loss.
- `camera_moved`: global scene motion better explains the disappearance.
- `camera_stopped_tracking`: the camera reduced tracking motion and the target was near the frame edge.
- `undetermined`: the evidence does not separate a dominant cause.

In the diagnostic plots, the solid red vertical line marks `decision_time_s`, the first robust kinematic acceleration onset before disappearance. The decision uses a longer pre-disappearance window so the baseline is not contaminated by the final accelerated phase. The code intentionally avoids choosing the largest late spike, because late spikes can come from image artifacts, target bloom, or thermal-pixel dissipation. The dotted red line marks `last_seen_time_s`, the last frame where the UAP was detected. The gray dashed line marks `missing_start_time_s`, the first frame where the UAP is already missing.

`camera_dx_px` and `camera_dy_px` are per-frame motion estimates. A long pan/tilt can look small there if each frame moves only a little. Use `camera_cumulative_dx_px` and `camera_cumulative_dy_px`, shown in `plot_axis_diagnostics(result)`, to see total camera displacement over time.

For camera-motion quality control, inspect `camera_motion_method`, `camera_phase_response`, `camera_phase_valid_fraction`, `camera_block_count`, `camera_block_score`, `camera_background_mask_coverage`, `camera_dense_valid_fraction`, and `camera_feature_mask_coverage` in `result["track"]`. For this type of fast pan, `camera_motion_method` should often be `phase_correlation` or `block_match`; if valid fractions or block counts are low, provide `roi` and `camera_motion_ignore_boxes` so the estimator sees more real background and less HUD/overlay.

`camera_dx_px` is the image/background displacement from one frame to the next. If the camera pans right, background texture often moves left, so `camera_dx_px` can be negative. Use `camera_pan_dx_px = -camera_dx_px` when you want the camera pointing direction convention.

Image coordinates use `y=0` at the top of the frame, so lower pixel `y` means visually higher in the image. The UAP Y plot is inverted to match this visual convention. Predicted UAP X/Y is plotted only while the target is still detected; after disappearance it is not an observed position.

`uap_x_px` and `uap_y_px` are observed image positions, so they include camera motion. To decide whether the UAP itself is moving mostly in X or Y, use `uap_residual_vx_px_s` and `uap_residual_vy_px_s`, which are plotted in `plot_axis_diagnostics(result)` after camera compensation.

The event table also includes `camera_after_cumulative_dx_px`, `camera_after_cumulative_dy_px`, and `camera_after_cumulative_translation_px`, which summarize camera displacement in the analysis window after the disappearance begins.

For a stronger analysis, tune `roi`, `object_bbox`, `min_blob_area`, `diff_threshold`, and `disappearance_patience`, then compare the charts generated by `plot_diagnostics` and `plot_axis_diagnostics`.

`plot_axis_diagnostics(result)` is useful when the camera and target move differently by axis. It plots camera `dx`, camera `dy`, cumulative camera motion, detected/predicted UAP `x`, detected/predicted UAP `y`, residual UAP `vx/vy`, and residual UAP speed. Use it to check cases where the UAP moves mostly horizontally while the camera moves vertically.